# Wegenregister Infrastructure Data Pipeline
**Source:** Flemish Road Register (Wegenregister) – `Wegsegment.shp`  
**Dictionary:** Objectcataloog WR v1.10 (18/02/2025)  

### What this notebook does
1. Loads the road-segment shapefile  
2. Standardises the coordinate reference system (EPSG:31370)  
3. Renames Dutch columns to English  
4. Cleans data (nulls, sentinel values, date parsing)  
5. Maps coded values to readable English labels  
6. Computes segment length in metres  
7. Produces a full English data dictionary  
8. Exports a clean GeoPackage + CSV ready for modelling  


In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import time, warnings, os
warnings.filterwarnings('ignore')
print('Libraries loaded.')


Libraries loaded.


## 1  Column Rename Map  (Dutch → English)
Source: *Objectcataloog WR*, Section 3.1 – Wegsegment field definitions.


In [2]:
# Dutch field name  ->  English field name
COLUMN_RENAME = {
    'WS_OIDN'  : 'segment_id',           # Unique object ID of the road segment
    'WS_UIDN'  : 'segment_version',       # Version identifier of the road segment
    'WS_GIDN'  : 'geometry_version',      # Version identifier of the segment geometry
    'B_WK_OIDN': 'start_node_id',         # Object ID of the start junction node
    'E_WK_OIDN': 'end_node_id',           # Object ID of the end junction node
    'STATUS'   : 'status_code',           # Segment lifecycle status code
    'LBLSTATUS': 'status_label_nl',       # Status label (Dutch)
    'MORF'     : 'morphology_code',       # Morphological road class code
    'LBLMORF'  : 'morphology_label_nl',   # Morphology label (Dutch)
    'WEGCAT'   : 'road_category_code',    # Road category (Spatial Structure Plan)
    'LBLWEGCAT': 'road_category_label_nl',# Road category label (Dutch)
    'LSTRNMID' : 'left_streetname_id',    # CRAB street-name code, left side
    'LSTRNM'   : 'left_streetname',       # Street name on the left side
    'RSTRNMID' : 'right_streetname_id',   # CRAB street-name code, right side
    'RSTRNM'   : 'right_streetname',      # Street name on the right side
    'BEHEER'   : 'manager_code',          # Code of the road-maintenance organisation
    'LBLBEHEER': 'manager_label',         # Full name of the road manager
    'METHODE'  : 'geometry_method_code',  # Geometry quality / acquisition method code
    'LBLMETHOD': 'geometry_method_label_nl',# Geometry method label (Dutch)
    'OPNDATUM' : 'record_date',           # Date segment first entered the database
    'BEGINTIJD': 'start_timestamp',       # Timestamp of database record creation
    'BEGINORG' : 'creator_org_code',      # Code of organisation that created the record
    'LBLBGNORG': 'creator_org_label',     # Label of creator organisation
    'TGBEP'    : 'access_code',           # Access restriction code
    'LBLTGBEP' : 'access_label_nl',       # Access restriction label (Dutch)
}
print(f'Column rename map ready: {len(COLUMN_RENAME)} columns.')


Column rename map ready: 25 columns.


## 2  Value Mapping Dictionaries  (code → English label)
Source: *Objectcataloog WR*, Section 4 – Code lists.


In [3]:
# ── 4.1  Segment Status ──────────────────────────────────────────────────────
STATUS_MAP = {
    1 : 'permit_requested',    # Road on an official document under review
    2 : 'permit_granted',      # Road on an approved, non-expired building permit
    3 : 'under_construction',  # Start of works has been reported
    4 : 'in_use',              # Works delivered; road is fully operational
    5 : 'out_of_use',          # Decommissioned but not demolished
    -8: 'unknown',             # No information available
}

# ── 4.2  Morphological Road Class ────────────────────────────────────────────
MORPHOLOGY_MAP = {
    101: 'motorway',                   # Dual carriageway, no at-grade crossings
    102: 'divided_road_non_motorway',  # Physically separated lanes, not motorway
    103: 'single_carriageway',         # One carriageway, traffic not separated
    104: 'roundabout',                 # Closed ring, one-way traffic only
    105: 'special_traffic_situation',  # Roughly circular, not a roundabout
    106: 'traffic_square',             # Unstructured area (market square, car park)
    107: 'ramp_grade_separated',       # On/off ramp at a grade-separated crossing
    108: 'ramp_at_grade',              # On/off ramp at an at-grade crossing
    109: 'parallel_road',              # Ramp with both ends on the same motorway
    110: 'service_road',               # Parallel to main road, separated by minor structure
    111: 'parking_entry_exit',         # Road designed to access a car park/garage
    112: 'service_entry_exit',         # Road to access a service facility
    113: 'pedestrian_zone',            # Exclusively for pedestrians
    114: 'cycleway_footpath',          # Pedestrians/cyclists only; width < 2.5 m
    116: 'tramway',                    # Exclusively for trams
    120: 'authorised_service_road',    # Exclusively for authorised services
    125: 'unpaved_road',               # Dirt road, no hard surface
    130: 'ferry',                      # Ferry route over water
    -8 : 'unknown',
}

# ── 4.3  Road Category (Flemish Spatial Structure Plan) ───────────────────
ROAD_CATEGORY_MAP = {
    'H'    : 'main_road',             # Links major urban areas; international connections
    'PI'   : 'primary_road_I',        # Complements main network; no transit function
    'PII'  : 'primary_road_II',       # Collector for areas of regional importance
    'PII-1': 'primary_road_II_type1', # Link/collector for entire urban area or port
    'PII-2': 'primary_road_II_type2', # Collector in regional/small city; possible ring
    'PII-3': 'primary_road_II_type3', # Collector for small city or tourist hub
    'S'    : 'secondary_road',        # Sub-municipal collector; connects to primary
    'L'    : 'local_road',            # Local road connecting to secondary network
    'EW'   : 'local_access_road',     # No through-traffic; direct plot access only
    '-9'   : 'not_applicable',
    '-8'   : 'unknown',
}

# ── 4.4  Geometry Acquisition Method ─────────────────────────────────────────
GEOMETRY_METHOD_MAP = {
    1: 'sketched',  # Geometry was sketched (lower positional accuracy)
    2: 'surveyed',  # Geometry derived from as-built plan or other dataset
}

# ── 4.5  Access Restriction ───────────────────────────────────────────────────
ACCESS_MAP = {
    1: 'public_road',             # Publicly accessible
    2: 'physically_inaccessible', # Blocked by obstacles
    3: 'legally_restricted',      # Access forbidden by law
    4: 'private_road',            # Limited access due to private ownership
    5: 'seasonal_access',         # Accessibility depends on season
    6: 'toll_road',               # Access subject to toll charges
}

print('All value mapping dictionaries ready.')


All value mapping dictionaries ready.


## 3  Load Shapefile


In [9]:
WEGSEGMENT_PATH = "../../data/extra/Wegenregister/Shapefile/Wegsegment.shp"

print(f'Loading: {WEGSEGMENT_PATH}')
print('This may take a few minutes for the full dataset...')
t0 = time.time()

wegen_gdf = gpd.read_file(WEGSEGMENT_PATH)

elapsed = time.time() - t0
print(f'Loaded {len(wegen_gdf):,} segments in {elapsed:.1f} s')
print(f'CRS : {wegen_gdf.crs}')
print(f'Columns ({len(wegen_gdf.columns)}): {wegen_gdf.columns.tolist()}')
wegen_gdf.head(2)


Loading: ../../data/extra/Wegenregister/Shapefile/Wegsegment.shp
This may take a few minutes for the full dataset...
Loaded 835,866 segments in 30.6 s
CRS : PROJCS["BD72 / Belgian Lambert 72",GEOGCS["BD72",DATUM["Reseau_National_Belge_1972",SPHEROID["International 1924",6378388,297,AUTHORITY["EPSG","7022"]],AUTHORITY["EPSG","6313"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",90],PARAMETER["central_meridian",4.36748666666667],PARAMETER["standard_parallel_1",49.8333339],PARAMETER["standard_parallel_2",51.1666672333333],PARAMETER["false_easting",150000.01256],PARAMETER["false_northing",5400088.4378],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Columns (26): ['WS_OIDN', 'WS_UIDN', 'WS_GIDN', 'B_WK_OIDN', 'E_WK_OIDN', 'STATUS', 'LBLSTATUS', 'MORF', 'LBLMORF', 'WEGCAT', 'LBLWEGCAT', 'LSTRNMID', 'LSTRNM', 'RSTRNMID', 'RSTRNM', 'BEHEER', 'LBLBEHEER', 'METHODE', 'LBLME

,WS_OIDN,WS_UIDN,WS_GIDN,B_WK_OIDN,E_WK_OIDN,STATUS,LBLSTATUS,MORF,LBLMORF,WEGCAT,...,LBLBEHEER,METHODE,LBLMETHOD,OPNDATUM,BEGINTIJD,BEGINORG,LBLBGNORG,TGBEP,LBLTGBEP,geometry
0,1,1_2,1_1,126722,41353,4,in gebruik,114,"wandel- of fietsweg, niet toegankelijk voor an...",-9,...,Stad Hasselt,2,ingemeten,20140220T143532,20250102T094713,AGIV,Agentschap voor Geografische Informatie Vlaand...,1,openbare weg,"LINESTRING (217368.75 181577.016, 217400.11 18..."
1,4,4_3,4_3,7,8,4,in gebruik,120,dienstweg,-9,...,District Centraal Limburg,2,ingemeten,20170315T154411,20170315T154448,AWV,Agentschap Wegen en Verkeer,1,openbare weg,"LINESTRING (232327.054 165044.681, 232319.001 ..."


## 4  Standardise CRS & Rename Columns


In [10]:
# 4a. Ensure EPSG:31370 (Belgian Lambert 72 – unit = metres)
TARGET_CRS = 'EPSG:31370'
if wegen_gdf.crs.to_epsg() != 31370:
    print('Converting CRS to EPSG:31370 ...')
    wegen_gdf = wegen_gdf.to_crs(TARGET_CRS)
else:
    print('CRS already EPSG:31370 – no conversion needed.')

# 4b. Rename Dutch columns to English
wegen_gdf = wegen_gdf.rename(columns=COLUMN_RENAME)
print('Columns after rename:', wegen_gdf.columns.tolist())


Converting CRS to EPSG:31370 ...
Columns after rename: ['segment_id', 'segment_version', 'geometry_version', 'start_node_id', 'end_node_id', 'status_code', 'status_label_nl', 'morphology_code', 'morphology_label_nl', 'road_category_code', 'road_category_label_nl', 'left_streetname_id', 'left_streetname', 'right_streetname_id', 'right_streetname', 'manager_code', 'manager_label', 'geometry_method_code', 'geometry_method_label_nl', 'record_date', 'start_timestamp', 'creator_org_code', 'creator_org_label', 'access_code', 'access_label_nl', 'geometry']


## 5  Clean Data


In [11]:
print('=== Before cleaning ===')
print(f'  Total rows      : {len(wegen_gdf):,}')
print(f'  Missing geometry: {wegen_gdf.geometry.isna().sum():,}')
print(f'  Status counts   :\n{wegen_gdf["status_code"].value_counts().to_string()}')

# 5a. Drop rows with missing geometry
wegen_gdf = wegen_gdf[wegen_gdf.geometry.notna()].copy()

# 5b. Keep only road segments that are 'in use' (status_code == 4)
wegen_gdf = wegen_gdf[wegen_gdf['status_code'] == 4].copy()

# 5c. Parse date/timestamp columns
for col in ['record_date', 'start_timestamp']:
    if col in wegen_gdf.columns:
        wegen_gdf[col] = pd.to_datetime(wegen_gdf[col], errors='coerce')

# 5d. Replace sentinel values (-8 = unknown, -9 = not applicable) with NaN
for col in ['status_code', 'morphology_code', 'geometry_method_code', 'access_code']:
    if col in wegen_gdf.columns:
        wegen_gdf[col] = wegen_gdf[col].replace({-8: np.nan, -9: np.nan})

# 5e. Replace -9 in ID columns with NaN
for col in ['left_streetname_id', 'right_streetname_id']:
    if col in wegen_gdf.columns:
        wegen_gdf[col] = wegen_gdf[col].replace({-9: np.nan})

# 5f. Compute segment length in metres
wegen_gdf['length_m'] = wegen_gdf.geometry.length.round(2)

print('\n=== After cleaning ===')
print(f'  Rows (in-use)      : {len(wegen_gdf):,}')
print(f'  Total road length  : {wegen_gdf["length_m"].sum()/1000:,.1f} km')


=== Before cleaning ===
  Total rows      : 835,866
  Missing geometry: 0
  Status counts   :
status_code
4    834974
3       394
2       259
1       152
5        87

=== After cleaning ===
  Rows (in-use)      : 834,974
  Total road length  : 104,442.6 km


## 6  Map Coded Values to English Labels


In [12]:
wegen_gdf['status_en']           = wegen_gdf['status_code'].map(STATUS_MAP)
wegen_gdf['morphology_en']       = wegen_gdf['morphology_code'].map(MORPHOLOGY_MAP)
wegen_gdf['road_category_en']    = wegen_gdf['road_category_code'].map(ROAD_CATEGORY_MAP)
wegen_gdf['geometry_method_en']  = wegen_gdf['geometry_method_code'].map(GEOMETRY_METHOD_MAP)
wegen_gdf['access_en']           = wegen_gdf['access_code'].map(ACCESS_MAP)

print('Distribution – morphology_en (top 10):')
print(wegen_gdf['morphology_en'].value_counts().head(10).to_string())

print('\nDistribution – road_category_en:')
print(wegen_gdf['road_category_en'].value_counts().to_string())

print('\nDistribution – access_en:')
print(wegen_gdf['access_en'].value_counts().to_string())


Distribution – morphology_en (top 10):
morphology_en
single_carriageway           546989
cycleway_footpath            121413
unpaved_road                 105558
divided_road_non_motorway     36504
roundabout                     8029
parking_entry_exit             4115
service_road                   2930
ramp_grade_separated           2704
motorway                       1253
tramway                        1207

Distribution – road_category_en:
road_category_en
local_access_road    510052
not_applicable       245839
unknown                1003

Distribution – access_en:
access_en
public_road                823913
private_road                10751
legally_restricted            250
physically_inaccessible        44
toll_road                      12
seasonal_access                 4


## 7  Select Final Columns for Modelling


In [13]:
FINAL_COLS = [
    'segment_id',
    'start_node_id',
    'end_node_id',
    'status_en',
    'morphology_code',
    'morphology_en',
    'road_category_code',
    'road_category_en',
    'access_code',
    'access_en',
    'geometry_method_en',
    'left_streetname',
    'right_streetname',
    'manager_code',
    'manager_label',
    'record_date',
    'length_m',
    'geometry',
]

final_cols = [c for c in FINAL_COLS if c in wegen_gdf.columns]
wegen_clean = wegen_gdf[final_cols].copy()

print(f'Final shape: {wegen_clean.shape}')
print(wegen_clean.dtypes)
wegen_clean.head(3)


Final shape: (834974, 18)
segment_id                     int64
start_node_id                  int64
end_node_id                    int64
status_en                     object
morphology_code              float64
morphology_en                 object
road_category_code            object
road_category_en              object
access_code                    int32
access_en                     object
geometry_method_en            object
left_streetname               object
right_streetname              object
manager_code                  object
manager_label                 object
record_date           datetime64[ns]
length_m                     float64
geometry                    geometry
dtype: object


,segment_id,start_node_id,end_node_id,status_en,morphology_code,morphology_en,road_category_code,road_category_en,access_code,access_en,geometry_method_en,left_streetname,right_streetname,manager_code,manager_label,record_date,length_m,geometry
0,1,126722,41353,in_use,114.0,cycleway_footpath,-9,not_applicable,1,public_road,surveyed,None,None,71072,Stad Hasselt,2014-02-20 14:35:32,83.60,"LINESTRING (217368.75 181577.016, 217400.11 18..."
1,4,7,8,in_use,120.0,authorised_service_road,-9,not_applicable,1,public_road,surveyed,None,None,AWV720,District Centraal Limburg,2017-03-15 15:44:11,243.62,"LINESTRING (232327.054 165044.681, 232319.001 ..."
2,6,650425,12,in_use,120.0,authorised_service_road,-9,not_applicable,1,public_road,surveyed,None,None,AWV720,District Centraal Limburg,2017-03-09 10:13:22,145.63,"LINESTRING (219742.688 177266.625, 219748.501 ..."


## 8  Data Quality Summary


In [14]:
missing = wegen_clean.isna().sum()
pct     = (missing / len(wegen_clean) * 100).round(1)
quality = pd.DataFrame({'missing_n': missing, 'missing_%': pct})
print('Missing values per column:')
display(quality[quality['missing_n'] > 0])

print('\nLength statistics (metres):')
display(wegen_clean['length_m'].describe().round(2).to_frame())


Missing values per column:


,missing_n,missing_%
morphology_code,3,0.0
morphology_en,3,0.0
road_category_en,78080,9.4
left_streetname,341919,40.9
right_streetname,342000,41.0



Length statistics (metres):


,length_m
count,834974.00
mean,125.08
std,167.53
min,1.02
25%,36.88
50%,78.77
75%,154.63
max,9948.49


## 9  English Data Dictionary
Full reference for all columns in `wegen_clean` plus code-list lookup tables.


In [15]:
DATA_DICT_ROWS = [
    ('segment_id',          'int',      'Unique object identifier for each road segment.',                                 '1, 4, 6'),
    ('start_node_id',       'int',      'Object identifier of the start junction node.',                                   '126722'),
    ('end_node_id',         'int',      'Object identifier of the end junction node.',                                     '41353'),
    ('status_en',           'str',      'Road segment lifecycle status (English). Always in_use after cleaning filter.',   'in_use'),
    ('morphology_code',     'int',      'Numeric code of the morphological road class (see MORPHOLOGY_MAP).',              '103, 114'),
    ('morphology_en',       'str',      'Morphological road class in English. Describes the physical shape/type.',         'single_carriageway, cycleway_footpath'),
    ('road_category_code',  'str',      'Alphanumeric road-category code from the Flemish Spatial Structure Plan.',        'H, PI, PII, S, L, EW, -9'),
    ('road_category_en',    'str',      'Road category in English. Indicates functional hierarchy in the network.',        'main_road, local_access_road'),
    ('access_code',         'int',      'Numeric code for public accessibility of the segment (see ACCESS_MAP).',          '1, 4'),
    ('access_en',           'str',      'Access restriction in English.',                                                  'public_road, private_road'),
    ('geometry_method_en',  'str',      'Method used to determine the geometry: surveyed (high quality) or sketched.',    'surveyed, sketched'),
    ('left_streetname',     'str',      'Official street name on the left side of the segment (CRAB register).',           'Bondgenotenlaan'),
    ('right_streetname',    'str',      'Official street name on the right side of the segment (CRAB register).',          'Bondgenotenlaan'),
    ('manager_code',        'str',      'Organisation code responsible for physical road maintenance.',                    'AWV, 71072, PARTIC'),
    ('manager_label',       'str',      'Full name of the road-management organisation.',                                  'Agentschap Wegen en Verkeer'),
    ('record_date',         'datetime', 'Date the segment was first entered into the Wegenregister database.',             '2014-02-20'),
    ('length_m',            'float',    'Length of the road segment in metres, computed from the geometry.',               '45.3'),
    ('geometry',            'LineString','Geospatial polyline in EPSG:31370 (Belgian Lambert 72, unit = metres).',         'LINESTRING (...)'),
]

dd = pd.DataFrame(DATA_DICT_ROWS, columns=['column', 'dtype', 'description', 'example'])
print('=== WEGENREGISTER ROAD SEGMENT – ENGLISH DATA DICTIONARY ===')
display(dd)

print('\n--- STATUS_MAP ---')
display(pd.DataFrame(list(STATUS_MAP.items()), columns=['code','status_en']))

print('\n--- MORPHOLOGY_MAP ---')
display(pd.DataFrame(list(MORPHOLOGY_MAP.items()), columns=['code','morphology_en']))

print('\n--- ROAD_CATEGORY_MAP ---')
display(pd.DataFrame(list(ROAD_CATEGORY_MAP.items()), columns=['code','road_category_en']))

print('\n--- ACCESS_MAP ---')
display(pd.DataFrame(list(ACCESS_MAP.items()), columns=['code','access_en']))

print('\n--- GEOMETRY_METHOD_MAP ---')
display(pd.DataFrame(list(GEOMETRY_METHOD_MAP.items()), columns=['code','geometry_method_en']))


=== WEGENREGISTER ROAD SEGMENT – ENGLISH DATA DICTIONARY ===


,column,dtype,description,example
0,segment_id,int,Unique object identifier for each road segment.,"1, 4, 6"
1,start_node_id,int,Object identifier of the start junction node.,126722
2,end_node_id,int,Object identifier of the end junction node.,41353
3,status_en,str,Road segment lifecycle status (English). Alway...,in_use
4,morphology_code,int,Numeric code of the morphological road class (...,"103, 114"
5,morphology_en,str,Morphological road class in English. Describes...,"single_carriageway, cycleway_footpath"
6,road_category_code,str,Alphanumeric road-category code from the Flemi...,"H, PI, PII, S, L, EW, -9"
7,road_category_en,str,Road category in English. Indicates functional...,"main_road, local_access_road"
8,access_code,int,Numeric code for public accessibility of the s...,"1, 4"
9,access_en,str,Access restriction in English.,"public_road, private_road"



--- STATUS_MAP ---


,code,status_en
0,1,permit_requested
1,2,permit_granted
2,3,under_construction
3,4,in_use
4,5,out_of_use
5,-8,unknown



--- MORPHOLOGY_MAP ---


...

,code,road_category_en
0,H,main_road
1,PI,primary_road_I
2,PII,primary_road_II
3,PII-1,primary_road_II_type1
4,PII-2,primary_road_II_type2
5,PII-3,primary_road_II_type3
6,S,secondary_road
7,L,local_road
8,EW,local_access_road
9,-9,not_applicable



--- ACCESS_MAP ---


,code,access_en
0,1,public_road
1,2,physically_inaccessible
2,3,legally_restricted
3,4,private_road
4,5,seasonal_access
5,6,toll_road



--- GEOMETRY_METHOD_MAP ---


,code,geometry_method_en
0,1,sketched
1,2,surveyed


## 10  Export Clean Data


In [16]:
OUTPUT_DIR = 'explo/andy'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GeoPackage – single file, preserves geometry and all dtypes
gpkg_path = os.path.join(OUTPUT_DIR, 'wegsegment_clean.gpkg')
wegen_clean.to_file(gpkg_path, driver='GPKG')
print(f'GeoPackage saved : {gpkg_path}')

# CSV – attribute table only (no geometry), for quick inspection
csv_path = os.path.join(OUTPUT_DIR, 'wegsegment_clean.csv')
wegen_clean.drop(columns='geometry').to_csv(csv_path, index=False)
print(f'CSV saved        : {csv_path}')

print(f'\nwegen_clean ready: {len(wegen_clean):,} rows x {len(wegen_clean.columns)} columns')
print('Use wegen_clean (or load the .gpkg) as your infrastructure feature table.')


GeoPackage saved : explo/andy\wegsegment_clean.gpkg
CSV saved        : explo/andy\wegsegment_clean.csv

wegen_clean ready: 834,974 rows x 18 columns
Use wegen_clean (or load the .gpkg) as your infrastructure feature table.


In [17]:
wegen_clean.head()

,segment_id,start_node_id,end_node_id,status_en,morphology_code,morphology_en,road_category_code,road_category_en,access_code,access_en,geometry_method_en,left_streetname,right_streetname,manager_code,manager_label,record_date,length_m,geometry
0,1,126722,41353,in_use,114.0,cycleway_footpath,-9,not_applicable,1,public_road,surveyed,None,None,71072,Stad Hasselt,2014-02-20 14:35:32,83.60,"LINESTRING (217368.75 181577.016, 217400.11 18..."
1,4,7,8,in_use,120.0,authorised_service_road,-9,not_applicable,1,public_road,surveyed,None,None,AWV720,District Centraal Limburg,2017-03-15 15:44:11,243.62,"LINESTRING (232327.054 165044.681, 232319.001 ..."
2,6,650425,12,in_use,120.0,authorised_service_road,-9,not_applicable,1,public_road,surveyed,None,None,AWV720,District Centraal Limburg,2017-03-09 10:13:22,145.63,"LINESTRING (219742.688 177266.625, 219748.501 ..."
3,7,41353,146626,in_use,114.0,cycleway_footpath,-9,not_applicable,1,public_road,surveyed,None,None,71072,Stad Hasselt,2014-02-20 14:35:32,5.74,"LINESTRING (217400.11 181499.516, 217403.479 1..."
4,8,593579,16,in_use,120.0,authorised_service_road,EHW,NaN,1,public_road,surveyed,None,None,AWV720,District Centraal Limburg,2017-08-24 14:58:33,187.35,"LINESTRING (200437.766 184002.969, 200414.083 ..."


In [18]:
wegen_clean['morphology_en'].value_counts()

morphology_en
single_carriageway           546989
cycleway_footpath            121413
unpaved_road                 105558
divided_road_non_motorway     36504
roundabout                     8029
parking_entry_exit             4115
service_road                   2930
ramp_grade_separated           2704
motorway                       1253
tramway                        1207
pedestrian_zone                1159
ramp_at_grade                  1117
authorised_service_road         925
service_entry_exit              551
parallel_road                   255
special_traffic_situation       198
traffic_square                   55
ferry                             9
Name: count, dtype: int64